In [ ]:
# ── Cell 0:切到工作目錄的專案根目錄 ──
# notebook 預設的 cwd 是它自己所在的資料夾(src/gesture_demo/),
# 這樣 import src.gesture_demo.* 會找不到(No module named 'src'),
# "data/raw/gestures.csv" 這種相對路徑也會不對。
# 往上找到含有 src/ 的那一層專案根,加進 sys.path 並切過去。
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("專案根目錄:", ROOT)

# ── Cell 1:import ──
import torch
import torch.nn as nn

from src.gesture_demo.model import GestureMLP
from src.gesture_demo.seeding import set_seed
# 超參數和 DataLoader 的建法都從 train.py 拿,notebook 和 script 共用同一份設定,
# 不會出現「notebook 跑 40 epoch、train.py 跑 30 epoch」這種對不起來的狀況。
from src.gesture_demo.train import CONFIG, build_loaders, save_run_metadata, MODEL_PATH

print("CONFIG =", CONFIG)


In [ ]:
# ── Cell 2:讀資料 + 切分 ──
# set_seed 要在建 DataLoader / 模型之前呼叫。
# notebook 的 cell 可以亂序重跑,所以「每個有隨機性的 cell」開頭都重設一次種子,
# 單獨重跑那個 cell 也會得到一樣的結果。
set_seed(CONFIG["seed"])

train_loader, test_loader = build_loaders(CONFIG)
print(f"train batches: {len(train_loader)}, test batches: {len(test_loader)}")


In [ ]:
# ── Cell 3:建模型 ──
# 權重初始化抽全域 torch RNG,所以這裡也要先重設種子,
# 否則單獨重跑這個 cell 就是一組全新的隨機權重。
set_seed(CONFIG["seed"])

model = GestureMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])


In [ ]:
# ── Cell 4:訓練 ──
for epoch in range(CONFIG["epochs"]):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1:2d}: loss = {total_loss/len(train_loader):.4f}")


In [ ]:
# ── Cell 5:評估 ──
from sklearn.metrics import confusion_matrix, classification_report
from src.gesture_demo.dataset import GESTURE_LABELS

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        idx = model(X_batch).argmax(dim=1)
        all_preds.extend(idx.tolist())
        all_labels.extend(y_batch.tolist())

acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"test accuracy: {acc:.4f}")
# labels 明確指定:某一類還沒資料 / 沒被切進 test 時也不會少一行
all_idx = list(range(len(GESTURE_LABELS)))
print(classification_report(all_labels, all_preds,
                            labels=all_idx, target_names=GESTURE_LABELS,
                            zero_division=0))


In [ ]:
# ── Cell 6:存模型(想存才跑)──
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), MODEL_PATH)
# 連同 seed / epochs / 資料檔 sha256 / git commit 一起記到 models/gesture_mlp.json,
# 事後才知道這個 .pth 是哪一次訓練的產物。
save_run_metadata(MODEL_PATH, CONFIG, acc, source="train.ipynb")
print("已存")
